## **https://www.kaggle.com/datasets/kartik2112/fraud-detection?select=fraudTrain.csv**

In [1]:
!pip install kaggle

In [2]:
from google.colab import files
files.upload()  # Upload kaggle.json
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d kartik2112/fraud-detection
!unzip fraud-detection.zip -d fraud_data

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/kartik2112/fraud-detection
License(s): CC0-1.0
100% 202M/202M [00:02<00:00, 81.0MB/s]

Archive:  fraud-detection.zip
  inflating: fraud_data/fraudTest.csv  
  inflating: fraud_data/fraudTrain.csv  


#### **Import Libraries**

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

#### **Load Dataset**

In [13]:
df = pd.read_csv("/content/fraud_data/fraudTrain.csv")

print(df.head())

   Unnamed: 0 trans_date_trans_time            cc_num  \
0           0   2019-01-01 00:00:18  2703186189652095   
1           1   2019-01-01 00:00:44      630423337322   
2           2   2019-01-01 00:00:51    38859492057661   
3           3   2019-01-01 00:01:16  3534093764340240   
4           4   2019-01-01 00:03:06   375534208663984   

                             merchant       category     amt      first  \
0          fraud_Rippin, Kub and Mann       misc_net    4.97   Jennifer   
1     fraud_Heller, Gutmann and Zieme    grocery_pos  107.23  Stephanie   
2                fraud_Lind-Buckridge  entertainment  220.11     Edward   
3  fraud_Kutch, Hermiston and Farrell  gas_transport   45.00     Jeremy   
4                 fraud_Keeling-Crist       misc_pos   41.96      Tyler   

      last gender                        street  ...      lat      long  \
0    Banks      F                561 Perry Cove  ...  36.0788  -81.1781   
1     Gill      F  43039 Riley Greens Suite 393  ...  48

In [14]:
print(df.shape)

(1296675, 23)


In [15]:
print(df['is_fraud'].value_counts())

is_fraud
0    1289169
1       7506
Name: count, dtype: int64


#### **Select Important Features**

In [5]:
data = df[['amt','category','state','unix_time','is_fraud']]

#### **Feature Engineering**

In [6]:
data = df[['amt','category','state','unix_time','is_fraud']].copy()

# Convert unix timestamp to datetime
data['datetime'] = pd.to_datetime(data['unix_time'], unit='s')

# Extract time-based features
data['hour'] = data['datetime'].dt.hour
data['day'] = data['datetime'].dt.day
data['dayofweek'] = data['datetime'].dt.dayofweek



In [7]:
# Drop unnecessary columns
data = data.drop(['unix_time', 'datetime'], axis=1)

#### **Encoding Categorical Features**

In [8]:
le_category = LabelEncoder()
le_state = LabelEncoder()

data['category'] = le_category.fit_transform(data['category'])
data['state'] = le_state.fit_transform(data['state'])

#### **Separate Features and Target**

In [9]:
X = data.drop("is_fraud", axis=1)
y = data["is_fraud"]

#### **Train Test Split**

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#### **Handle Class Imbalance with SMOTE**

In [11]:
smote = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", pd.Series(y_train_resampled).value_counts())

Before SMOTE: is_fraud
0    1031335
1       6005
Name: count, dtype: int64
After SMOTE: is_fraud
0    1031335
1    1031335
Name: count, dtype: int64


#### **Feature Scaling**

In [12]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_resampled)
X_test_scaled = scaler.transform(X_test)